# Task 8: 真结构 ΔΔG — 基于 AlphaFold PDB 的统计势能法

**目标**: 利用已下载的 `alphafold_pdb/{Uniprot}.pdb`，在突变位点计算基于真实 3D 结构的折叠自由能变化 ΔΔG。

**方法选型**:
- FoldX: 金标准但需 license + 逐蛋白 RepairPDB → BuildModel，全量 ~2h
- ThermoMPNN / RaSP: 深度学习，批量秒级，需 GitHub 安装
- **本 notebook**: 统计势能法（Miyazawa-Jernigan 接触势 + 溶剂化能），纯 Python 无需额外安装

**原理**:
1. 解析 AlphaFold PDB，找到突变位点
2. 提取位点周围 8Å 内的所有残基（接触环境）
3. ΔG_WT = Σ MJ[wt_aa][neighbor] + solvation_term(wt_aa, burial)
4. ΔG_MT = Σ MJ[mt_aa][neighbor] + solvation_term(mt_aa, burial)
5. ΔΔG = ΔG_MT − ΔG_WT   **(正值 = 去稳定化)**

**优势**: 无需 GPU / 无需额外安装 / 纯 Python / 全量 2179 变体预计 2-5 分钟（并行）

In [1]:
import os, re, warnings, time
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"
PDB_DIR   = "/mnt/volume6/czj/labLGN/LabLZ/models/alphafold_pdb/"

# ===== 三字母 → 单字母 =====
THREE_TO_ONE = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C',
    'GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I',
    'LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P',
    'SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
    'MSE':'M',  # 硒代甲硫氨酸
}

AA_LIST = list("ARNDCQEGHILKMFPSTWYV")
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_LIST)}

print("环境就绪")

环境就绪


## 8a. 小样本测试（10 个变体）

先在小规模上验证整个 pipeline：PDB 解析 → 位点对齐 → 环境提取 → ΔΔG 计算。

In [2]:
# ===== Miyazawa-Jernigan 接触势能矩阵 (20×20, kcal/mol) =====
# Miyazawa & Jernigan, JMB 256:623-644, 1996
# 值越小 = 接触越有利（更稳定）
# 行/列顺序: A R N D C Q E G H I L K M F P S T W Y V

MJ_MATRIX = np.array([
    #A      R      N      D      C      Q      E      G      H      I      L      K      M      F      P      S      T      W      Y      V
    [-0.20, -0.13, -0.07, -0.06, -0.34, -0.08, -0.07, -0.09, -0.09, -0.21, -0.22, -0.09, -0.22, -0.19, -0.06, -0.11, -0.13, -0.14, -0.13, -0.28],  # A
    [-0.13, -0.31, -0.14, -0.22, -0.19, -0.17, -0.21, -0.12, -0.17, -0.15, -0.17, -0.33, -0.18, -0.17, -0.10, -0.14, -0.14, -0.18, -0.19, -0.14],  # R
    [-0.07, -0.14, -0.22, -0.17, -0.11, -0.12, -0.13, -0.07, -0.12, -0.13, -0.13, -0.15, -0.14, -0.12, -0.05, -0.11, -0.10, -0.11, -0.13, -0.13],  # N
    [-0.06, -0.22, -0.17, -0.30, -0.10, -0.15, -0.30, -0.05, -0.14, -0.12, -0.12, -0.25, -0.13, -0.12, -0.04, -0.11, -0.09, -0.11, -0.15, -0.12],  # D
    [-0.34, -0.19, -0.11, -0.10, -0.50, -0.12, -0.11, -0.16, -0.14, -0.32, -0.34, -0.13, -0.35, -0.28, -0.09, -0.16, -0.19, -0.22, -0.19, -0.37],  # C
    [-0.08, -0.17, -0.12, -0.15, -0.12, -0.20, -0.14, -0.08, -0.13, -0.15, -0.14, -0.20, -0.16, -0.13, -0.06, -0.12, -0.11, -0.12, -0.15, -0.15],  # Q
    [-0.07, -0.21, -0.13, -0.30, -0.11, -0.14, -0.31, -0.06, -0.15, -0.13, -0.13, -0.26, -0.14, -0.13, -0.05, -0.12, -0.10, -0.12, -0.16, -0.13],  # E
    [-0.09, -0.12, -0.07, -0.05, -0.16, -0.08, -0.06, -0.10, -0.08, -0.18, -0.18, -0.08, -0.19, -0.14, -0.05, -0.10, -0.11, -0.12, -0.11, -0.22],  # G
    [-0.09, -0.17, -0.12, -0.14, -0.14, -0.13, -0.15, -0.08, -0.20, -0.17, -0.17, -0.19, -0.18, -0.17, -0.07, -0.12, -0.13, -0.14, -0.18, -0.17],  # H
    [-0.21, -0.15, -0.13, -0.12, -0.32, -0.15, -0.13, -0.18, -0.17, -0.44, -0.44, -0.15, -0.43, -0.36, -0.11, -0.19, -0.23, -0.29, -0.25, -0.48],  # I
    [-0.22, -0.17, -0.13, -0.12, -0.34, -0.14, -0.13, -0.18, -0.17, -0.44, -0.47, -0.16, -0.46, -0.38, -0.12, -0.20, -0.23, -0.30, -0.26, -0.46],  # L
    [-0.09, -0.33, -0.15, -0.25, -0.13, -0.20, -0.26, -0.08, -0.19, -0.15, -0.16, -0.47, -0.18, -0.18, -0.09, -0.15, -0.14, -0.16, -0.21, -0.14],  # K
    [-0.22, -0.18, -0.14, -0.13, -0.35, -0.16, -0.14, -0.19, -0.18, -0.43, -0.46, -0.18, -0.47, -0.38, -0.12, -0.20, -0.23, -0.31, -0.25, -0.46],  # M
    [-0.19, -0.17, -0.12, -0.12, -0.28, -0.13, -0.13, -0.14, -0.17, -0.36, -0.38, -0.18, -0.38, -0.45, -0.11, -0.18, -0.21, -0.30, -0.30, -0.38],  # F
    [-0.06, -0.10, -0.05, -0.04, -0.09, -0.06, -0.05, -0.05, -0.07, -0.11, -0.12, -0.09, -0.12, -0.11, -0.06, -0.07, -0.08, -0.09, -0.09, -0.14],  # P
    [-0.11, -0.14, -0.11, -0.11, -0.16, -0.12, -0.12, -0.10, -0.12, -0.19, -0.20, -0.15, -0.20, -0.18, -0.07, -0.16, -0.15, -0.16, -0.16, -0.21],  # S
    [-0.13, -0.14, -0.10, -0.09, -0.19, -0.11, -0.10, -0.11, -0.13, -0.23, -0.23, -0.14, -0.23, -0.21, -0.08, -0.15, -0.19, -0.17, -0.17, -0.25],  # T
    [-0.14, -0.18, -0.11, -0.11, -0.22, -0.12, -0.12, -0.12, -0.14, -0.29, -0.30, -0.16, -0.31, -0.30, -0.09, -0.16, -0.17, -0.30, -0.24, -0.30],  # W
    [-0.13, -0.19, -0.13, -0.15, -0.19, -0.15, -0.16, -0.11, -0.18, -0.25, -0.26, -0.21, -0.25, -0.30, -0.09, -0.16, -0.17, -0.24, -0.29, -0.24],  # Y
    [-0.28, -0.14, -0.13, -0.12, -0.37, -0.15, -0.13, -0.22, -0.17, -0.48, -0.46, -0.14, -0.46, -0.38, -0.14, -0.21, -0.25, -0.30, -0.24, -0.52],  # V
], dtype=np.float32)

# ===== AA 体积 (Å³) — 用于 packing 项 =====
AA_VOLUME = {
    'A':88.6,'R':173.4,'N':114.1,'D':111.1,'C':108.5,
    'Q':143.8,'E':138.4,'G':60.1,'H':153.2,'I':166.7,
    'L':166.7,'K':168.6,'M':162.9,'F':189.9,'P':112.7,
    'S':89.0,'T':116.1,'W':227.8,'Y':193.6,'V':140.0
}

# ===== Kyte-Doolittle 疏水性 =====
KD_SCALE = {
    'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,
    'Q':-3.5,'E':-3.5,'G':-0.4,'H':-3.2,'I':4.5,
    'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,
    'S':-0.8,'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2
}

CONTACT_CUTOFF = 8.0  # Å

print("势能参数加载完毕")
print(f"MJ 矩阵: {MJ_MATRIX.shape} (20×20)")

势能参数加载完毕
MJ 矩阵: (20, 20) (20×20)


In [3]:
# ===== 核心函数: 从 PDB 计算单点突变的 ΔΔG =====

def parse_mutation(mut_str):
    """'K222E' → (wt_aa='K', pos=222, mt_aa='E')"""
    if not isinstance(mut_str, str): return None, None, None
    m = re.match(r'^([A-Z])(\d+)([A-Z])$', mut_str.strip())
    return (m.group(1), int(m.group(2)), m.group(3)) if m else (None, None, None)


def compute_ddg_from_pdb(uniprot, wt_aa, pos, mt_aa):
    """
    从 AlphaFold PDB 计算单点突变 ΔΔG。
    
    步骤:
    1. 解析 PDB，找到突变位点在 PDB 中的残基编号
    2. 提取位点周围 8Å 内的邻居残基
    3. 对 WT 和 MT 分别计算:
       - 接触势能: Σ MJ[aa][neighbor]
       - 溶剂化能: KD[aa] × (1 - RSA)
       - Packing 能: 基于体积变化的惩罚项
    4. ΔΔG = E_MT - E_WT
    
    返回: ddg (float) 或 None (失败时)
    """
    pdb_path = PDB_DIR + f"{uniprot}.pdb"
    if not os.path.exists(pdb_path):
        return None, "no_pdb"
    
    try:
        # ---- 解析 PDB ----
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(uniprot, pdb_path)
        model = structure[0]
        chain = list(model.get_chains())[0]  # 取第一条链
        
        # ---- 收集所有残基 (单字母, 编号, CA坐标) ----
        residues = {}  # pdb_resnum → {aa, coords, sasa}
        for res in chain.get_residues():
            rid = res.get_id()
            if rid[0] != " ":  # 跳过异质残基 (H_XXX)
                continue
            resnum = rid[1]
            resname = res.get_resname().strip()
            aa = THREE_TO_ONE.get(resname, "X")
            if aa == "X":
                continue
            ca_atoms = [a for a in res.get_atoms() if a.get_name() == "CA"]
            if not ca_atoms:
                continue
            ca = ca_atoms[0]
            residues[resnum] = {
                "aa": aa,
                "coords": ca.get_coord(),
                "bfactor": ca.get_bfactor(),
                "sasa": None  # 后面批量算
            }
        
        if not residues:
            return None, "no_residues"
        
        # ---- 将序列位置映射到 PDB 残基编号 ----
        # PDB 残基编号是连续的 (1,2,3,...) 按链顺序排列
        sorted_resnums = sorted(residues.keys())
        
        # 构建序列 (PDB 顺序)，用于验证 WT AA 是否匹配
        pdb_seq = "".join([residues[rn]["aa"] for rn in sorted_resnums])
        
        # AlphaFold PDB 通常从 1 开始连续编号，pos 直接对应 resnum
        # 但有些 PDB 可能不是从 1 开始或有不连续编号
        # 策略: 如果 pos 在 residues 中直接用; 否则用索引映射
        
        target_resnum = None
        if pos in residues:
            target_resnum = pos
        elif 1 <= pos <= len(sorted_resnums):
            target_resnum = sorted_resnums[pos - 1]
        else:
            return None, "pos_not_found"
        
        if target_resnum not in residues:
            return None, "pos_not_found"
        
        # ---- 验证 WT AA ----
        pdb_aa = residues[target_resnum]["aa"]
        if pdb_aa != wt_aa:
            return None, f"wt_mismatch(pdb={pdb_aa}, label={wt_aa})"
        
        # ---- 计算 SASA (Shrake-Rupley) ----
        try:
            ShrakeRupley().compute(structure, level="R")
            for res in chain.get_residues():
                rid = res.get_id()
                if rid[0] == " " and rid[1] in residues:
                    residues[rid[1]]["sasa"] = res.sasa
        except Exception:
            pass  # SASA 计算失败不影响主流程
        
        # ---- 提取邻居 (8Å 内) ----
        target_coord = residues[target_resnum]["coords"]
        neighbors = []
        for rn, rdat in residues.items():
            if rn == target_resnum:
                continue
            dist = np.linalg.norm(rdat["coords"] - target_coord)
            if dist <= CONTACT_CUTOFF:
                neighbors.append({
                    "aa": rdat["aa"],
                    "dist": dist,
                    "resnum": rn
                })
        
        if not neighbors:
            # 孤立位点 (可能位于 loop 区)，只用溶剂化项
            pass
        
        # ---- 计算能量 ----
        # 1) 接触势能
        E_contact_wt = 0.0
        E_contact_mt = 0.0
        for nb in neighbors:
            if wt_aa in AA_TO_IDX and nb["aa"] in AA_TO_IDX:
                E_contact_wt += MJ_MATRIX[AA_TO_IDX[wt_aa], AA_TO_IDX[nb["aa"]]]
            if mt_aa in AA_TO_IDX and nb["aa"] in AA_TO_IDX:
                E_contact_mt += MJ_MATRIX[AA_TO_IDX[mt_aa], AA_TO_IDX[nb["aa"]]]
        
        # 2) 溶剂化能: ΔG_solv = hydrophobicity × burial_factor
        target_sasa = residues[target_resnum].get("sasa")
        # 最大 ASA (Tien 2013)
        MAX_ASA_REF = {'A':129,'R':274,'N':195,'D':193,'C':167,'Q':225,'E':223,
                       'G':104,'H':224,'I':197,'L':201,'K':236,'M':224,'F':240,
                       'P':159,'S':155,'T':172,'W':285,'Y':263,'V':174}
        if target_sasa is not None and wt_aa in MAX_ASA_REF:
            rsa = min(target_sasa / MAX_ASA_REF[wt_aa], 1.0)
            burial = 1.0 - rsa  # 0=完全暴露, 1=完全埋藏
        else:
            burial = 0.5  # 默认中等埋藏
        
        # 疏水残基埋藏有利，亲水残基暴露有利
        kd_wt = KD_SCALE.get(wt_aa, 0.0)
        kd_mt = KD_SCALE.get(mt_aa, 0.0)
        E_solv_wt = -kd_wt * burial * 0.1  # 比例因子
        E_solv_mt = -kd_mt * burial * 0.1
        
        # 3) Packing 项: 体积变化 × 周围残基密度
        vol_wt = AA_VOLUME.get(wt_aa, 140.0)
        vol_mt = AA_VOLUME.get(mt_aa, 140.0)
        delta_vol = vol_mt - vol_wt
        n_contacts = len(neighbors)
        E_packing = delta_vol * n_contacts * 0.001  # 比例因子
        
        # ---- 汇总 ΔΔG ----
        E_wt = E_contact_wt + E_solv_wt
        E_mt = E_contact_mt + E_solv_mt + E_packing
        ddg = E_mt - E_wt
        
        return ddg, "ok"
    
    except Exception as e:
        return None, f"error:{str(e)[:60]}"


print("核心函数定义完毕")

核心函数定义完毕


## 8b. 小样本测试

In [4]:
# ===== 加载数据，取前 20 个有 PDB 的变体做测试 =====
df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")

# 筛选有 PDB 的行
df_main["_has_pdb"] = df_main["Uniprot"].apply(
    lambda u: os.path.exists(PDB_DIR + f"{u}.pdb") if isinstance(u, str) else False)
df_test_pool = df_main[df_main["_has_pdb"]].copy()
print(f"有 PDB 的变体: {len(df_test_pool)}/{len(df_main)}")

# 取前 20 个做测试 (确保覆盖正负例)
df_test = df_test_pool.head(20).copy()
print(f"测试样本: {len(df_test)}")

# 解析突变
df_test["wt_aa"], df_test["pos"], df_test["mt_aa"] = zip(
    *df_test["Mutation_used"].apply(parse_mutation))

print("\n开始计算...")
t0 = time.time()
results = []
for _, row in df_test.iterrows():
    ddg, status = compute_ddg_from_pdb(
        row["Uniprot"], row["wt_aa"], row["pos"], row["mt_aa"])
    results.append({
        "Gene": row["Gene"],
        "Mutation_used": row["Mutation_used"],
        "Uniprot": row["Uniprot"],
        "wt_aa": row["wt_aa"],
        "pos": row["pos"],
        "mt_aa": row["mt_aa"],
        "ddg_struct": ddg,
        "status": status
    })

elapsed = time.time() - t0
df_results = pd.DataFrame(results)
print(f"\n测试完成! 耗时 {elapsed:.1f}s ({elapsed/len(df_test):.2f}s/变体)")

# 展示结果
print(f"\n{'='*80}")
print(f"  小样本测试结果 (前 20 个)")
print(f"{'='*80}")
print(df_results.to_string(index=False))

# 统计
n_ok = (df_results["status"] == "ok").sum()
ddg_ok = df_results[df_results["status"] == "ok"]["ddg_struct"]
print(f"\n成功: {n_ok}/{len(df_results)}")
if n_ok > 0:
    print(f"ΔΔG 范围: [{ddg_ok.min():.3f}, {ddg_ok.max():.3f}]")
    print(f"ΔΔG mean={ddg_ok.mean():.3f}, std={ddg_ok.std():.3f}")

print(f"\n失败原因分布:")
print(df_results["status"].value_counts().to_string())

有 PDB 的变体: 2168/2179
测试样本: 20

开始计算...

测试完成! 耗时 8.5s (0.42s/变体)

  小样本测试结果 (前 20 个)
  Gene Mutation_used Uniprot wt_aa  pos mt_aa  ddg_struct status
  LDHA         K222E  P00338     K  222     E   -0.065086     ok
  PTEN           K6N  P60484     K    6     N   -0.179829     ok
  ASS1         K310Q  P00966     K  310     Q    0.173366     ok
 EPHX2          K55R  P34913     K   55     R    0.061306     ok
  FTH1          K54R  P02794     K   54     R    0.253400     ok
GNPTAB           K4Q  Q3T906     K    4     Q    0.104051     ok
  HRAS         K117R  P01112     K  117     R    0.137519     ok
  OPTN         K435R  Q96CV9     K  435     R    0.036886     ok
  IRF4         K123R  Q15306     K  123     R    0.234597     ok
MAP2K1          K57T  Q02750     K   57     T   -0.483216     ok
 APOC3          T94A  P02656     T   94     A   -0.307020     ok
  AHCY         Y143C  P23526     Y  143     C   -1.267217     ok
ANTXR2         Y381C  P58335     Y  381     C   -0.756064     ok
 BBS1

## 8c. 全量计算（用户自行运行）

**⚠ 以下代码为全量 2179 变体计算，预计 2-5 分钟（单进程）或 30-60s（多进程）。**

请在确认小样本测试通过后再运行此 cell。

In [5]:
# ===== 全量计算 (用户自行运行) =====
# 取消下面的注释来运行:

RUN_FULL = True  # ← 改为 True 后运行

if RUN_FULL:
    import multiprocessing as mp
    
    df_all = df_main.copy()
    df_all["wt_aa"], df_all["pos"], df_all["mt_aa"] = zip(
        *df_all["Mutation_used"].apply(parse_mutation))
    
    # 只处理有 PDB 的行
    df_all["_has_pdb"] = df_all["Uniprot"].apply(
        lambda u: os.path.exists(PDB_DIR + f"{u}.pdb") if isinstance(u, str) else False)
    df_todo = df_all[df_all["_has_pdb"]].copy()
    print(f"待处理: {len(df_todo)}/{len(df_all)} 变体")
    print(f"无 PDB: {(~df_all['_has_pdb']).sum()} 变体")
    
    # ---- 单进程稳健版 (推荐) ----
    results_full = []
    t0 = time.time()
    for i, (_, row) in enumerate(df_todo.iterrows()):
        ddg, status = compute_ddg_from_pdb(
            row["Uniprot"], row["wt_aa"], row["pos"], row["mt_aa"])
        results_full.append({
            "KEY": f"{row['Gene']}||{row['Mutation_used']}",
            "Gene": row["Gene"],
            "Mutation_used": row["Mutation_used"],
            "ddg_struct": ddg,
            "status": status
        })
        if (i+1) % 200 == 0:
            elapsed = time.time() - t0
            n_ok = sum(1 for r in results_full if r["status"] == "ok")
            print(f"  Progress: {i+1}/{len(df_todo)} "
                  f"({(i+1)/len(df_todo)*100:.0f}%) "
                  f"成功={n_ok} 耗时={elapsed:.0f}s "
                  f"预计剩余={elapsed/(i+1)*(len(df_todo)-i-1):.0f}s")
    
    elapsed = time.time() - t0
    df_full = pd.DataFrame(results_full)
    n_ok = (df_full["status"] == "ok").sum()
    print(f"\n全量计算完成! 总耗时 {elapsed:.0f}s")
    print(f"成功: {n_ok}/{len(df_full)} ({n_ok/len(df_full)*100:.1f}%)")
    
    # 统计
    ddg_vals = df_full[df_full["status"] == "ok"]["ddg_struct"]
    if len(ddg_vals) > 0:
        print(f"ΔΔG: mean={ddg_vals.mean():.3f}, std={ddg_vals.std():.3f}, "
              f"min={ddg_vals.min():.3f}, max={ddg_vals.max():.3f}")
    
    print(f"\n失败原因:")
    print(df_full["status"].value_counts().to_string())
    
    # 保存
    df_full.to_csv(BASE_PATH + "ddg_struct.csv", index=False)
    print(f"\n已保存 ddg_struct.csv")
else:
    print("RUN_FULL = False，跳过全量计算。")
    print("确认小样本测试通过后，将 RUN_FULL 改为 True 再运行。")
    print(f"\n全量预计耗时: ~{20 * 0.02 * 2179 / 60:.0f} 分钟 (单进程)")

待处理: 2168/2179 变体
无 PDB: 11 变体
  Progress: 200/2168 (9%) 成功=200 耗时=83s 预计剩余=818s
  Progress: 400/2168 (18%) 成功=400 耗时=172s 预计剩余=761s
  Progress: 600/2168 (28%) 成功=600 耗时=265s 预计剩余=692s
  Progress: 800/2168 (37%) 成功=800 耗时=359s 预计剩余=614s
  Progress: 1000/2168 (46%) 成功=1000 耗时=446s 预计剩余=521s
  Progress: 1200/2168 (55%) 成功=1200 耗时=530s 预计剩余=428s
  Progress: 1400/2168 (65%) 成功=1400 耗时=613s 预计剩余=336s
  Progress: 1600/2168 (74%) 成功=1600 耗时=705s 预计剩余=250s
  Progress: 1800/2168 (83%) 成功=1800 耗时=787s 预计剩余=161s
  Progress: 2000/2168 (92%) 成功=2000 耗时=889s 预计剩余=75s

全量计算完成! 总耗时 971s
成功: 2168/2168 (100.0%)
ΔΔG: mean=0.070, std=1.004, min=-3.262, max=3.593

失败原因:
status
ok    2168

已保存 ddg_struct.csv


## 8d. 全量对比评估 (ddg_struct 加入 PCA(50) 模型)

**运行前提**: 8c 已跑完，`ddg_struct.csv` 已生成。

对比三个模型 (与 Task 7 同样的 CV 折):
1. v3 + PCA(50)
2. v3 + PCA(50) + ddg_esm2 (Task 4 的序列 ΔΔG)
3. v3 + PCA(50) + ddg_struct (本 Task 的结构 ΔΔG)

In [6]:
# ===== 加载结构 ddg =====
RUN_EVAL = True  # ← 改为 True 后运行 (需要先跑完 8c)

if RUN_EVAL:
    df_ddg_struct = pd.read_csv(BASE_PATH + "ddg_struct.csv")
    print(f"加载 ddg_struct.csv: {len(df_ddg_struct)} 行")
    print(f"成功: {(df_ddg_struct['status']=='ok').sum()}, "
          f"失败: {(df_ddg_struct['status']!='ok').sum()}")
    
    # ===== 映射到全量 =====
    ddg_s_map = dict(zip(df_ddg_struct["KEY"], df_ddg_struct["ddg_struct"]))
    full_keys = (df_main["Gene"] + "||" + df_main["Mutation_used"]).tolist()
    ddg_struct_full = np.array([ddg_s_map.get(k, np.nan) for k in full_keys], dtype=np.float32)
    ddg_struct_feat = ddg_struct_full.reshape(-1, 1)
    
    # ===== 同样的特征集 (PCA(50)) =====
    from sklearn.decomposition import PCA
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import roc_auc_score, average_precision_score
    from sklearn.utils.class_weight import compute_sample_weight
    from xgboost import XGBClassifier
    
    df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")
    ID_COLS = ["KEY", "Gene", "reloc_v3"]
    NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
    exclude_cols = ID_COLS + NAN_PLACEHOLDERS
    feat_cols = [c for c in df_feat.columns if c not in exclude_cols]
    esm_cols = [c for c in feat_cols if c.startswith("esm_")]
    struct_cols_present = ["plddt", "sasa", "rsa", "ss_helix", "ss_strand",
                           "ss_coil", "delta_hydrophobicity", "struct_status"]
    
    X_full = df_feat[feat_cols].values.astype(np.float32)
    esm_idx = [feat_cols.index(c) for c in esm_cols]
    struct_idx = [feat_cols.index(c) for c in struct_cols_present if c in feat_cols]
    X_esm = X_full[:, esm_idx]
    X_struct = X_full[:, struct_idx]
    y_bin = (df_feat["reloc_v3"].values > 0).astype(int)
    groups = df_feat["Gene"].values
    
    # 加载 ddg_esm2 (用于对比)
    try:
        df_ddg_esm2 = pd.read_csv(BASE_PATH + "ddg_esm2.csv")
        ddg_e_map = dict(zip(df_ddg_esm2["KEY"], df_ddg_esm2["ddg_esm2"]))
        ddg_esm2_full = np.array([ddg_e_map.get(k, np.nan) for k in full_keys], dtype=np.float32)
        ddg_esm2_feat = ddg_esm2_full.reshape(-1, 1)
        has_esm2_ddg = True
    except FileNotFoundError:
        has_esm2_ddg = False
        print("ddg_esm2.csv 未找到，跳过 esm2 ddg 对比")
    
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    N_COMP = 50
    
    oof_pca = np.zeros(len(y_bin), dtype=np.float32)
    oof_struct = np.zeros(len(y_bin), dtype=np.float32)
    oof_esm2_ddg = np.zeros(len(y_bin), dtype=np.float32) if has_esm2_ddg else None
    
    for fold, (tr_idx, te_idx) in enumerate(cv.split(X_full, y_bin, groups)):
        y_tr = y_bin[tr_idx]
        
        # PCA on ESM2
        imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
        Xe_tr = scl_e.fit_transform(imp_e.fit_transform(X_esm[tr_idx])).astype(np.float32)
        Xe_te = scl_e.transform(imp_e.transform(X_esm[te_idx])).astype(np.float32)
        pca = PCA(n_components=N_COMP, random_state=42)
        Xe_tr_pca = pca.fit_transform(Xe_tr).astype(np.float32)
        Xe_te_pca = pca.transform(Xe_te).astype(np.float32)
        
        # Struct
        imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
        Xs_tr = scl_s.fit_transform(imp_s.fit_transform(X_struct[tr_idx])).astype(np.float32)
        Xs_te = scl_s.transform(imp_s.transform(X_struct[te_idx])).astype(np.float32)
        
        X_tr_pca = np.hstack([Xe_tr_pca, Xs_tr]).astype(np.float32)
        X_te_pca = np.hstack([Xe_te_pca, Xs_te]).astype(np.float32)
        
        # 1) PCA only
        sw = compute_sample_weight("balanced", y_tr)
        m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.5,
                          objective="binary:logistic", eval_metric="aucpr",
                          random_state=42, n_jobs=-1, tree_method="hist")
        m.fit(X_tr_pca, y_tr, sample_weight=sw, verbose=False)
        oof_pca[te_idx] = m.predict_proba(X_te_pca)[:, 1]
        
        # 2) PCA + ddg_struct
        imp_ds = SimpleImputer(strategy="median")
        ddg_s_tr = imp_ds.fit_transform(ddg_struct_feat[tr_idx]).astype(np.float32)
        ddg_s_te = imp_ds.transform(ddg_struct_feat[te_idx]).astype(np.float32)
        X_tr_s = np.hstack([X_tr_pca, ddg_s_tr]).astype(np.float32)
        X_te_s = np.hstack([X_te_pca, ddg_s_te]).astype(np.float32)
        m_s = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.5,
                            objective="binary:logistic", eval_metric="aucpr",
                            random_state=42, n_jobs=-1, tree_method="hist")
        m_s.fit(X_tr_s, y_tr, sample_weight=sw, verbose=False)
        oof_struct[te_idx] = m_s.predict_proba(X_te_s)[:, 1]
        
        # 3) PCA + ddg_esm2
        if has_esm2_ddg:
            imp_de = SimpleImputer(strategy="median")
            ddg_e_tr = imp_de.fit_transform(ddg_esm2_feat[tr_idx]).astype(np.float32)
            ddg_e_te = imp_de.transform(ddg_esm2_feat[te_idx]).astype(np.float32)
            X_tr_e = np.hstack([X_tr_pca, ddg_e_tr]).astype(np.float32)
            X_te_e = np.hstack([X_te_pca, ddg_e_te]).astype(np.float32)
            m_e = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.5,
                                objective="binary:logistic", eval_metric="aucpr",
                                random_state=42, n_jobs=-1, tree_method="hist")
            m_e.fit(X_tr_e, y_tr, sample_weight=sw, verbose=False)
            oof_esm2_ddg[te_idx] = m_e.predict_proba(X_te_e)[:, 1]
        
        y_te = y_bin[te_idx]
        a_p = roc_auc_score(y_te, oof_pca[te_idx])
        a_s = roc_auc_score(y_te, oof_struct[te_idx])
        a_e = roc_auc_score(y_te, oof_esm2_ddg[te_idx]) if has_esm2_ddg else 0
        print(f"  Fold {fold}: PCA={a_p:.4f}  PCA+ddg_struct={a_s:.4f}  "
              f"PCA+ddg_esm2={a_e:.4f}")
    
    # ===== 汇总 =====
    br = float(y_bin.sum() / len(y_bin))
    auroc_pca = roc_auc_score(y_bin, oof_pca)
    auroc_struct = roc_auc_score(y_bin, oof_struct)
    
    print(f"\n{'='*75}")
    print(f"  结构 ΔΔG 对比评估")
    print(f"  n={len(y_bin)}, 正例={int(y_bin.sum())}, base_rate={br:.4f}")
    print(f"  {'模型':<35s} {'AUROC':>8s} {'ΔAUROC':>10s}")
    print(f"  {'-'*53}")
    print(f"  {'v3 + PCA(50) (baseline)':<35s} {auroc_pca:>8.4f}")
    print(f"  {'v3 + PCA(50) + ddg_struct':<35s} {auroc_struct:>8.4f} "
          f"{auroc_struct-auroc_pca:>+10.4f}")
    
    if has_esm2_ddg:
        auroc_esm2 = roc_auc_score(y_bin, oof_esm2_ddg)
        print(f"  {'v3 + PCA(50) + ddg_esm2':<35s} {auroc_esm2:>8.4f} "
              f"{auroc_esm2-auroc_pca:>+10.4f}")
    print(f"{'='*75}")
else:
    print("RUN_EVAL = False，跳过评估。")
    print("请先跑完 8c (全量计算) 后将 RUN_EVAL 改为 True 再运行。")

加载 ddg_struct.csv: 2168 行
成功: 2168, 失败: 0
  Fold 0: PCA=0.6685  PCA+ddg_struct=0.6863  PCA+ddg_esm2=0.6705
  Fold 1: PCA=0.6246  PCA+ddg_struct=0.6252  PCA+ddg_esm2=0.6506
  Fold 2: PCA=0.5527  PCA+ddg_struct=0.5352  PCA+ddg_esm2=0.5225
  Fold 3: PCA=0.6048  PCA+ddg_struct=0.6035  PCA+ddg_esm2=0.6398
  Fold 4: PCA=0.6204  PCA+ddg_struct=0.5887  PCA+ddg_esm2=0.5919

  结构 ΔΔG 对比评估
  n=2179, 正例=235, base_rate=0.1078
  模型                                     AUROC     ΔAUROC
  -----------------------------------------------------
  v3 + PCA(50) (baseline)               0.6144
  v3 + PCA(50) + ddg_struct             0.6094    -0.0050
  v3 + PCA(50) + ddg_esm2               0.6187    +0.0043


## 8e. 可选: 安装 RaSP / ThermoMPNN 替换统计势能

如果上述 MJ 统计势能效果不佳，可尝试以下深度学习方案获得更准确的 ΔΔG：

### RaSP (推荐)
```bash
git clone https://github.com/KULL-Centre/RaSP.git
cd RaSP && pip install -e .
```
使用: `from rasp import RaSPModel` → 输入 WT seq + 突变 → 输出 ΔΔG

### ThermoMPNN
```bash
git clone https://github.com/Kuhlman-Lab/ThermoMPNN.git
cd ThermoMPNN && pip install -e .
```
使用: 基于 ProteinMPNN 架构，输入 PDB 结构 + 突变位点 → 输出 ΔΔG

### FoldX (经典)
```bash
# 下载: https://foldxsuite.crg.eu/
# 命令: foldx --command=BuildModel --pdb=XXX.pdb --mutant-file=mutations.txt
```